In [45]:
from importlib.metadata import version

import langchain
import langchain_community
import langchain_core
import langchain_openai
import numpy as np
import transformers
from langchain_community.agent_toolkits.load_tools import load_tools as lc_load_tools
from langchain_core.runnables import RunnableSequence
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

print(version("langchain"))
print(version("langchain-core"))
print(version("langchain-openai"))
print(version("langchain-community"))

print("langchain_v", langchain.__version__)
print("langchain_core", langchain_core.__version__)
print("langchain_community", langchain_community.__version__)
print("transformers", transformers.__version__)
print("numpy", np.__version__)

import os
from langchain_openai import AzureChatOpenAI

from dotenv import load_dotenv
load_dotenv("../.env")

2.7.0+cu128
True
12.8
NVIDIA GeForce RTX 3080 Ti
1.2.3
1.3.0
1.1.14
0.4.1
langchain_v 1.2.3
langchain_core 1.3.0
langchain_community 0.4.1
transformers 4.40.2
numpy 1.26.4


True

### google/flan-t5-base

In [30]:
hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    temperature=0.0,
    device = 0 if torch.cuda.is_available() else -1
)

In [31]:
llm = HuggingFacePipeline(pipeline=hf_pipeline)
tools = lc_load_tools(["llm-math", "wikipedia"], llm=llm)

for t in tools:
    print(t.name, "->", t.description)

Calculator -> Useful for when you need to answer questions about math.
wikipedia -> A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.


In [32]:
def get_tool_by_keyword(keyword):
    for t in tools:
        if keyword.lower() in t.name.lower():
            return t
    return None

In [33]:
def agent(query: str):
    q = query.lower()

    # Math routing
    if any(x in q for x in ["*", "+", "-", "/", "calculate", "math"]):
        tool = get_tool_by_keyword("math") or get_tool_by_keyword("calc")
        if tool:
            return tool.run(query)

    # Wikipedia routing
    if any(x in q for x in ["who", "what", "where", "capital", "wiki"]):
        tool = get_tool_by_keyword("wiki")
        if tool:
            return tool.run(query)

    # Fallback
    return llm.invoke(query)

In [34]:
agent("What is the capital of France?")

'Page: Closed-ended question\nSummary: A closed-ended question is any question for which a researcher provides research participants with options from which to choose a response. Closed-ended questions are sometimes phrased as a statement that requires a response.\nA closed-ended question contrasts with an open-ended question, which cannot easily be answered with specific information.\n\n\n\nPage: Capital city\nSummary: A capital city, or just capital, is the municipality holding primary status in a country, state, province, department, or other subnational division, usually as its seat of government. A capital is typically a city that physically encompasses the government\'s offices and meeting places; the status as capital is often designated by law or a constitution. In some jurisdictions, including several countries, different branches of government are in different settlements, sometimes meaning there are multiple official capitals. In some cases, a distinction is made between the

In [35]:
agent("Explain quantum computing")

'quantum computers'

In [36]:
#Limited brief response as the model we used is very small
agent("paris!, thats interesting, any thoughts about paris, give me information related to source etc etc.")

'Paris is a city in France.'

### tiiuae/falcon-7b-instruct

In [38]:
#Using a bigger Model & test the same steps as above or continue with base model for now
# hf_pipeline = pipeline(
#     "text-generation",
#     model="tiiuae/falcon-7b-instruct",
#     device=0,
#     max_new_tokens=256,
#     temperature=0.0,
# )
# llm_model = HuggingFacePipeline(pipeline=hf_pipeline)

# tools = lc_load_tools(["llm-math", "wikipedia"], llm=llm_model)

In [39]:
#The steps here might fail as we are using newer version of Langchain which doesnt have
#create_react_agent, AgentExecutor etc..Thus commenting out
'''
from langchain_classic.agents import create_react_agent, AgentExecutor
from langsmith import Client

# ReAct prompt
prompt = Client().pull_prompt("hwchase17/react")

# Agent
#agent = create_react_agent(llm=llm_model, tools=tools, prompt=prompt)
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
# Executor
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

'''

'\nfrom langchain_classic.agents import create_react_agent, AgentExecutor\nfrom langsmith import Client\n\n# ReAct prompt\nprompt = Client().pull_prompt("hwchase17/react")\n\n# Agent\n#agent = create_react_agent(llm=llm_model, tools=tools, prompt=prompt)\nagent = create_react_agent(llm=llm, tools=tools, prompt=prompt)\n# Executor\nexecutor = AgentExecutor(\n    agent=agent,\n    tools=tools,\n    verbose=True,\n    handle_parsing_errors=True\n)\n'

In [40]:
'''question = "What is the capital of France and what is 17*23?"
result = executor.invoke({"input": question})
print(result["output"])
'''

'question = "What is the capital of France and what is 17*23?"\nresult = executor.invoke({"input": question})\nprint(result["output"])\n'

In [41]:
#If versions werent downgraded , continue with this option
'''
def agent_router(query):
    if any(op in query for op in ["+", "-", "*", "/", "calculate"]):
        tool = next(t for t in tools if "Calculator" in t.name)
        return tool.run(query)
    elif any(k in query.lower() for k in ["who", "what", "where", "capital", "wiki"]):
        tool = next(t for t in tools if "Wikipedia" in t.name)
        return tool.run(query)
    else:
        return llm.invoke(query)

question = "What is the capital of France"
print(agent_router(question))
'''

'\ndef agent_router(query):\n    if any(op in query for op in ["+", "-", "*", "/", "calculate"]):\n        tool = next(t for t in tools if "Calculator" in t.name)\n        return tool.run(query)\n    elif any(k in query.lower() for k in ["who", "what", "where", "capital", "wiki"]):\n        tool = next(t for t in tools if "Wikipedia" in t.name)\n        return tool.run(query)\n    else:\n        return llm.invoke(query)\n\nquestion = "What is the capital of France"\nprint(agent_router(question))\n'

### working with GPT model

In [47]:
#Two options,
#Downgrading & working with GPT model(older model say Gpt-4)
'''
!pip install -U \
  langchain==0.3.14 \
  langchain-core==0.3.32 \
  langchain-community==0.3.14 \
  langchain-openai==0.2.14 \
  pydantic==2.9.2
'''

'\n!pip install -U   langchain==0.3.14   langchain-core==0.3.32   langchain-community==0.3.14   langchain-openai==0.2.14   pydantic==2.9.2\n'

In [48]:
llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    # temperature=0,
)

In [49]:
#from langchain_community.agent_toolkits.load_tools import load_tools as lc_load_tools
tools = lc_load_tools(["llm-math", "wikipedia"], llm=llm)

'''
#works with current LangChain / LangSmith packages
from langsmith import Client
prompt = Client().pull_prompt("hwchase17/react")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

executor.invoke(
    {"input": "What is the capital of France"}
)
'''

#Getting ur custom functions
def get_tool_by_keyword(keyword):
    for t in tools:
        if keyword.lower() in t.name.lower():
            return t
    return None


def agent(query: str):
    q = query.lower()

    # Math routing
    if any(x in q for x in ["*", "+", "-", "/", "calculate", "math"]):
        tool = get_tool_by_keyword("math") or get_tool_by_keyword("calc")
        if tool:
            return tool.run(query)

    # Wikipedia routing
    if any(x in q for x in ["who", "what", "where", "capital", "wiki"]):
        tool = get_tool_by_keyword("wiki")
        if tool:
            return tool.run(query)

    # Fallback
    return llm.invoke(query)

In [50]:
agent("Capital of France?")

'Page: List of capitals of France\nSummary: This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944.\n\nPage: Capital punishment in France\nSummary: Capital punishment in France (French: peine de mort en France) is banned by Article 66-1 of the Constitution of the French Republic, voted as a constitutional amendment by the Congress of the French Parliament on 19 February 2007 and simply stating "No one can be sentenced to the death penalty" (French: Nul ne peut être condamné à la peine de mort). The death penalty was already declared illegal on 9 October 1981 when President François Mitterrand signed a law prohibiting the judicial system from using it and commuting the sentences of the seven people on death row to life imprisonment. The last execution took place by guillotine, being the main legal method since the French Revolution; Hamida Djandoubi, a Tunisian citizen convicted of torture and murder on French soil, was put 

In [51]:
agent("Is it the tool or llm answering this question")

AIMessage(content="It's me, the language model (LLM), answering your question! When you interact with this platform, your inputs are processed by the LLM, which generates the responses you see. Tools might be integrated alongside the LLM in some systems to perform specific tasks (like calculations, database queries, etc.), but the answers themselves come from the LLM.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 17, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 7, 'engine_ttft_ms': 35, 'engine_ttlt_ms': 554, 'pre_inference_ms': 86, 'service_tbt_ms': 7, 'service_ttft_ms': 277, 'service_ttlt_ms': 808, 'total_duration_ms': 729, 'user_visible_ttft_ms': 190}}, 'model_provider': 'openai', 'model_name': 'gpt